# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第8章 知覚実験・刺激作成・PsychoPy ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch08_trial_structure.R`

対象章: 第8章「知覚実験・刺激作成・PsychoPy」

「PsychoPyによる実装」節の図解。同定課題の1試行が、時間軸上でどのような
区間(注視点→刺激提示→反応取得→試行間間隔)に分かれるかを示す模式図。
実測データではなく、試行構造そのものを説明するための概念図である。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

segments <- tibble(
  phase = c("注視点", "刺激提示", "反応取得(反応時間の起点)", "試行間間隔(ITI)"),
  start = c(0, 500, 700, 700 + 2000),
  end   = c(500, 700, 700 + 2000, 700 + 2000 + 500),
  y     = 1
) |>
  mutate(phase = factor(phase, levels = phase))

label_df <- segments |>
  mutate(mid = (start + end) / 2)

p <- ggplot(segments) +
  geom_rect(aes(xmin = start, xmax = end, ymin = 0.3, ymax = 0.7, fill = phase), colour = "white") +
  geom_text(data = label_df, aes(x = mid, y = 0.5, label = phase), size = 3, colour = "white", fontface = "bold") +
  geom_segment(data = label_df, aes(x = start, xend = start, y = 0.25, yend = 0.3), colour = "grey40") +
  geom_text(data = label_df, aes(x = start, y = 0.15, label = paste0(start, "ms")), size = 2.8, colour = "grey30") +
  annotate("text", x = tail(segments$end, 1), y = 0.15,
           label = paste0(tail(segments$end, 1), "ms(次の試行開始)"), size = 2.8, colour = "grey30", hjust = 1) +
  scale_fill_manual(values = c("注視点" = "#8C8C8C", "刺激提示" = "#34518A",
                                "反応取得(反応時間の起点)" = "#A8681F", "試行間間隔(ITI)" = "#B7B7B7"),
                     guide = "none") +
  scale_x_continuous(limits = c(-50, tail(segments$end, 1) + 50)) +
  scale_y_continuous(limits = c(0, 1)) +
  theme_void() +
  labs(caption = "模式図(時間の目盛りは説明用の一例)。scripts/R/ch08_trial_structure.R より生成。") +
  theme(plot.caption = element_text(size = rel(0.8), colour = "grey35"))

save_fig(p, "fig08-02-trial-structure", chapter = "ch08", width = 7, height = 2.4)
cat("完了。図はassets/figures/ch08/に保存した。\n")


## `ch08_headphone_check.R`

対象章: 第8章「知覚実験・刺激作成・PsychoPy」

「オンライン実験とその計測上の制約」節の作例。位相を利用したヘッドホン
チェックで、何試行中何試行以上正解を基準とすれば、スピーカー使用者
(正答はほぼ偶然、3択なので正答率1/3と仮定)を十分な確率で不合格に
できるかを、二項分布に基づいて実際に計算する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_trials <- 6
p_headphone <- 0.90   # ヘッドホン使用者が単試行で正答する確率(仮定値)
p_speaker   <- 1 / 3  # スピーカー使用者が単試行で正答する確率(3択の偶然水準)

for (pass_threshold in c(4, 5, 6)) {
  pass_rate_headphone <- 1 - pbinom(pass_threshold - 1, n_trials, p_headphone)
  pass_rate_speaker   <- 1 - pbinom(pass_threshold - 1, n_trials, p_speaker)
  cat(n_trials, "試行中", pass_threshold, "試行以上正解を合格基準とする場合:\n")
  cat("  ヘッドホン使用者の合格率:", round(pass_rate_headphone * 100, 1), "%\n")
  cat("  スピーカー使用者が誤って合格する確率:", round(pass_rate_speaker * 100, 1), "%\n\n")
}


## `ch08_dprime_worked_example.R`

対象章: 第8章「知覚実験・刺激作成・PsychoPy」

「信号検出理論の基礎」節の作例。ヒット率とフォールスアラーム率から
d'(感度)とc(反応バイアス)を実際に計算する、具体的な数値例。
架空のデータであり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

hit_rate <- 0.82
fa_rate  <- 0.28

z_hit <- qnorm(hit_rate)
z_fa  <- qnorm(fa_rate)

d_prime <- z_hit - z_fa
criterion_c <- -0.5 * (z_hit + z_fa)

cat("ヒット率:", hit_rate, " フォールスアラーム率:", fa_rate, "\n")
cat("z(ヒット率) =", round(z_hit, 3), "  z(フォールスアラーム率) =", round(z_fa, 3), "\n")
cat("d' =", round(d_prime, 3), "\n")
cat("c  =", round(criterion_c, 3), "\n")

# 同じd'でも反応バイアスが異なる2人目の聞き手(cが大きい、慎重な反応)
hit_rate_2 <- 0.65
fa_rate_2  <- pnorm(qnorm(hit_rate_2) - d_prime)
cat("\n同じ感度(d' =", round(d_prime, 3), ")で、より慎重な反応基準を持つ聞き手の例:\n")
cat("ヒット率:", round(hit_rate_2, 3), " フォールスアラーム率:", round(fa_rate_2, 3), "\n")


## `ch08_sdt_roc.R`

対象章: 第8章「知覚実験・刺激作成・PsychoPy」

「信号検出理論の基礎」節の図解。ROC空間(横軸フォールスアラーム率、
縦軸ヒット率)上に、感度d'を一定に保ったまま反応バイアス(基準c)だけを
動かしたときの理論的なROC曲線と、ある1点の観測値を重ね、d'とcが
それぞれ何を表しているかを図示する。等分散正規分布を仮定する標準的な
SDTの定式化に基づく理論図であり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

d_prime_example <- 1.5

# 基準cを動かしたときのROC曲線(理論値、Macmillan & Creelmanの標準的な定式化):
# z(H) = d'/2 - c, z(F) = -d'/2 - c
c_seq <- seq(-3, 3, length.out = 300)
roc_curve <- tibble(
  c = c_seq,
  F = pnorm(-d_prime_example / 2 - c_seq),
  H = pnorm(d_prime_example / 2 - c_seq)
)

# 観測値の例(基準がやや保守的に偏っている場合)
c_obs <- 0.3
obs <- tibble(F = pnorm(-d_prime_example / 2 - c_obs), H = pnorm(d_prime_example / 2 - c_obs))
d_prime_obs <- qnorm(obs$H) - qnorm(obs$F)
c_obs_est   <- -0.5 * (qnorm(obs$H) + qnorm(obs$F))

p <- ggplot(roc_curve, aes(F, H)) +
  geom_abline(slope = 1, intercept = 0, linetype = 3, colour = "grey50") +
  geom_line(colour = "#34518A", linewidth = 1) +
  geom_point(data = obs, aes(F, H), size = 3.5, colour = "#A8681F") +
  geom_segment(data = obs, aes(x = F, xend = F, y = 0, yend = H), linetype = 2, colour = "#A8681F", linewidth = 0.4) +
  geom_segment(data = obs, aes(x = 0, xend = F, y = H, yend = H), linetype = 2, colour = "#A8681F", linewidth = 0.4) +
  annotate("text", x = 0.62, y = 0.35, label = "d'一定のROC曲線\n(反応バイアスcだけが変化)",
           size = 3.2, colour = "#34518A", hjust = 0) +
  annotate("text", x = obs$F + 0.03, y = obs$H - 0.08,
           label = paste0("観測値の例\nF=", round(obs$F, 2), ", H=", round(obs$H, 2),
                           "\nd'=", round(d_prime_obs, 2), ", c=", round(c_obs_est, 2)),
           size = 3, colour = "#A8681F", hjust = 0) +
  scale_x_continuous(limits = c(0, 1), labels = scales::percent, name = "フォールスアラーム率(F)") +
  scale_y_continuous(limits = c(0, 1), labels = scales::percent, name = "ヒット率(H)") +
  labs(caption = "理論図(等分散正規分布モデル)。scripts/R/ch08_sdt_roc.R より生成。")

save_fig(p, "fig08-03-sdt-roc", chapter = "ch08", width = 5.5, height = 5)
cat("d'(観測点から逆算):", round(d_prime_obs, 3), " c(観測点から逆算):", round(c_obs_est, 3), "\n")
cat("完了。図はassets/figures/ch08/に保存した。\n")
